In [1]:
!nvidia-smi

Mon Mar 23 09:44:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch transformers peft trl bitsandbytes accelerate datasets sentencepiece scipy evaluate rouge_score nltk streamlit huggingface_hub pandas numpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 123.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b484461fa4331472f91cbf98c35e21945800dc0400d5b47b52b1e43b739029f4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from huggingface_hub import login
login(token="your_huggingface_token")

In [11]:
from datasets import load_dataset

# Financial QA dataset - perfect for your project
dataset = load_dataset("virattt/financial-qa-10K")

print(dataset)
print(f"Train samples: {len(dataset['train'])}")
print(dataset['train'][0])

README.md:   0%|          | 0.00/419 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'context', 'ticker', 'filing'],
        num_rows: 7000
    })
})
Train samples: 7000
{'question': 'What area did NVIDIA initially focus on before expanding to other computationally intensive fields?', 'answer': 'NVIDIA initially focused on PC graphics.', 'context': 'Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.', 'ticker': 'NVDA', 'filing': '2023_10K'}


In [12]:
from datasets import Dataset

def format_sample(sample):
    return f"""### Instruction:
Answer the financial question based on the given context

### Input:
Context: {sample['context']}
Question: {sample['question']}

### Response:
{sample['answer']}"""

def prepare_dataset(data):
    formatted = [{"text": format_sample(d)} for d in data]
    return Dataset.from_list(formatted)

# Split train into train and test
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_dataset = prepare_dataset(split['train'])
test_dataset = prepare_dataset(split['test'])

print(f"Train samples: {len(train_dataset)} ✅")
print(f"Test samples: {len(test_dataset)} ✅")
print("\nFormatted Sample:")
print(train_dataset[0]['text'])

Train samples: 6300 ✅
Test samples: 700 ✅

Formatted Sample:
### Instruction:
Answer the financial question based on the given context

### Input:
Context: Highlights during fiscal year 2023 include the following: We generated $18,085 million of cash from operations.
Question: What was the amount of cash generated from operations by the company in fiscal year 2023?

### Response:
The company generated $18,085 million of cash from operations in fiscal year 2023.


In [25]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded successfully ✅")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully ✅


In [26]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [27]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="financial-qa-model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",        # renamed from evaluation_strategy
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

print("Training arguments set ✅")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments set ✅


In [23]:
import trl
print(trl.__version__)

0.29.1


In [29]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="financial-qa-model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,        # changed to False
    bf16=True,         # added this
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting training... ⏳")
trainer.train()
print("Training completed ✅")

Adding EOS to train dataset:   0%|          | 0/6300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6300 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6300 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Starting training... ⏳


Step,Training Loss,Validation Loss
100,0.983858,0.974008
200,0.903605,0.954133
300,0.937039,0.941255
400,0.974112,0.933886
500,0.918262,0.929344
600,0.923550,0.926306
700,0.828558,0.922025
800,0.788646,0.920030
900,0.845218,0.919823
1000,0.811542,0.919242


Training completed ✅


In [30]:
trainer.save_model("financial-qa-model")
tokenizer.save_pretrained("financial-qa-model")
print("Model saved locally ✅")

Model saved locally ✅


In [31]:
import torch

model.eval()

def generate_answer(question, context):
    prompt = f"""### Instruction:
Answer the financial question based on the given context

### Input:
Context: {context}
Question: {question}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# Test with sample
result = generate_answer(
    question="What is the total revenue mentioned?",
    context="The company reported total revenue of USD 2.5 billion for the fiscal year 2024, representing a 15% increase compared to the previous year"
)
print("Answer:", result)

Answer: The total revenue for the fiscal year 2024 was USD 2.5 billion.


In [32]:
from rouge_score import rouge_scorer
import torch

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

rouge1_scores = []
rougeL_scores = []

# Test on 20 samples
for sample in split['test'].select(range(20)):
    prompt = f"""### Instruction:
Answer the financial question based on the given context

### Input:
Context: {sample['context']}
Question: {sample['question']}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = prediction.split("### Response:")[-1].strip()
    reference = sample['answer']

    scores = scorer.score(reference, prediction)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

    print(f"Question  : {sample['question']}")
    print(f"Expected  : {reference}")
    print(f"Predicted : {prediction}")
    print(f"ROUGE-1: {scores['rouge1'].fmeasure:.4f} | ROUGE-L: {scores['rougeL'].fmeasure:.4f}")
    print("-" * 60)

print(f"\nAverage ROUGE-1: {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"Average ROUGE-L: {sum(rougeL_scores)/len(rougeL_scores):.4f}")

Question  : How much were the company's debt obligations as of December 31, 2023?
Expected  : $2,299,887 thousand
Predicted : $2,299,887 thousand
ROUGE-1: 1.0000 | ROUGE-L: 1.0000
------------------------------------------------------------
Question  : What are the specific structures and legal considerations for a management services organization (MSO) in relation to its relationship with physician owners?
Expected  : Management services organizations (MSOs) must structure their fees to comply with state fee splitting laws and often use stock transfer restriction agreements and/or other relationships to maintain a 'friendly' relationship with physician owners of the professional corporation.
Predicted : A management services organization (MSO) is a corporation that provides management services to a physician entity, and it is required to comply with state fee splitting laws. The fees under the MSO arrangement must be structured as a percentage-based fee, and the fees may be subject to

In [ ]:
from huggingface_hub import HfApi, create_repo

HF_USERNAME = "your_huggingface_username"   # e.g. "john123"
HF_TOKEN = "your_huggingface_token"         # e.g. "hf_xxxxxxxxxxxx"
REPO_NAME = "financial-qa-tinyllama"

# Create repo first
create_repo(
    repo_id=f"{HF_USERNAME}/{REPO_NAME}",
    token=HF_TOKEN,
    exist_ok=True
)

# Upload model
api = HfApi()
api.upload_folder(
    folder_path="financial-qa-model",
    repo_id=f"{HF_USERNAME}/{REPO_NAME}",
    repo_type="model",
    token=HF_TOKEN
)

print(f"Model pushed successfully ✅")
print(f"Visit: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-1000/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-1182/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...adapter_model.safetensors:   1%|1         |  211kB / 18.0MB            

...

  ...adapter_model.safetensors:   1%|1         |  106kB / 9.03MB            

  ...ckpoint-1182/optimizer.pt:   1%|1         |  213kB / 18.2MB            

  ...ckpoint-1182/scheduler.pt:   1%|1         |  17.0B / 1.47kB            

  ...nt-1182/training_args.bin:   1%|1         |  65.0B / 5.58kB            

Model pushed successfully ✅
Visit: https://huggingface.co/manireddys/financial-qa-tinyllama
